<a href="https://colab.research.google.com/github/FARNHELL/ML-INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FARNHELL/ML-INTERNSHIP/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**1.Unit of analysis:** One row = one content-item-day (a single page on a single calendar day).

**2.Tables used:** fact_content_daily_performance (March 2026 partition) + dim_content (static content metadata).

**3.Time window:** report_date in March 2026 (mid-panel month for iteration; June 2026 is sealed test month).

**4.Label / proxy:** Forward-window decline — will this content's impressions drop >20 % in the next 30 days?

**5.Deliberately excluded:** trend_direction and trend_pct from the starter CSV — they are label-derived (computed from impression ratios), not observed outcomes. Using them as features is leakage.

In [11]:
# Contract: Load March 2026 partition + dim_content
import os
import numpy as np
import pandas as pd
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN").strip()
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.environ["HF_TOKEN"].strip()
from huggingface_hub import hf_hub_download

mar_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)
df = pd.read_parquet(mar_path)
df["report_date"] = pd.to_datetime(df["report_date"])

dim_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset",
    token=hf_token
)
dim_df = pd.read_parquet(dim_path)

print(f"fact_content_daily_performance (Mar 2026): {len(df):,} rows")
print(f"dim_content: {len(dim_df):,} rows")
print(f"Date range: {df['report_date'].min().date()} -> {df['report_date'].max().date()}")
print(f"Distinct clients: {df['client_hash_id'].nunique()}")
print(f"Distinct content items: {df['content_hash_id'].nunique()}")

fact_content_daily_performance (Mar 2026): 9,841,378 rows
dim_content: 519,606 rows
Date range: 2026-03-01 -> 2026-03-31
Distinct clients: 55
Distinct content items: 331437


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
**Feature** — knowable BEFORE the moment you predict:

    gsc_impressions, gsc_clicks, gsc_avg_position (7-day trailing avg)
    ga4_pageviews, ga4_sessions (7-day trailing avg, when ga4_data_available is TRUE)
    scroll_events (7-day trailing avg)
    content_type (from dim_content, static)

**Label / proxy** — the thing you predict:

    forward_decline — binary: do impressions drop >20 % in the next 30 days?

**Context** — for grouping / joining only, never learned from:

    report_date, client_hash_id, content_hash_id

**Excluded** — private, future info, or label-derived:

    trend_direction, trend_pct (starter CSV) — label-derived, leakage
    gsc_data_available, ga4_data_available — flags used to filter, not features
    sessions_organic, sessions_direct, etc. — traffic-source breakdowns overlap with impression label; keeping them would let the model learn the label's composition


In [12]:
# Verify: show which columns are in each bucket
feature_cols = ["gsc_impressions", "gsc_clicks", "gsc_avg_position",
                "ga4_pageviews", "ga4_sessions", "scroll_events", "content_type"]
label_cols = ["forward_decline"]
context_cols = ["report_date", "client_hash_id", "content_hash_id"]
excluded_cols = ["trend_direction", "trend_pct", "gsc_data_available",
                 "ga4_data_available", "sessions_organic", "sessions_direct",
                 "sessions_referral", "sessions_social", "sessions_paid", "sessions_ai"]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Label ({len(label_cols)}): {label_cols}")
print(f"Context ({len(context_cols)}): {context_cols}")
print(f"Excluded ({len(excluded_cols)}): {excluded_cols}")

Features (7): ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'scroll_events', 'content_type']
Label (1): ['forward_decline']
Context (3): ['report_date', 'client_hash_id', 'content_hash_id']
Excluded (10): ['trend_direction', 'trend_pct', 'gsc_data_available', 'ga4_data_available', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
# Query 1 — Grain: (report_date, client_hash_id, content_hash_id) must be unique
dupes = df.groupby(["report_date", "client_hash_id", "content_hash_id"]).size()
dupes = dupes[dupes > 1]
print(f"Grain check — duplicate (date, client, content) tuples: {len(dupes)}")
if len(dupes) == 0:
    print("PASS: grain holds — one row per content-item-day.")
else:
    print("FAIL: grain violated — investigate duplicates.")

Grain check — duplicate (date, client, content) tuples: 0
PASS: grain holds — one row per content-item-day.


In [14]:
# Query 2 — Row count + date span for March 2026
print(f"Row count (Mar 2026): {len(df):,}")
print(f"Date span: {df['report_date'].min().date()} -> {df['report_date'].max().date()}")
print(f"Expected: ~9.8M rows, 2026-03-01 to 2026-03-31")

Row count (Mar 2026): 9,841,378
Date span: 2026-03-01 -> 2026-03-31
Expected: ~9.8M rows, 2026-03-01 to 2026-03-31


In [15]:
# Query 3 — Availability: filter with IS TRUE, show how many rows survive
gsc_available = df[df["gsc_data_available"].isin([True, 1])]
ga4_available = df[df["ga4_data_available"].isin([True, 1])]
both_available = df[df["gsc_data_available"].isin([True, 1]) & df["ga4_data_available"].isin([True, 1])]

print(f"gsc_data_available IS TRUE: {len(gsc_available):,} rows ({len(gsc_available)/len(df)*100:.1f} %)")
print(f"ga4_data_available IS TRUE: {len(ga4_available):,} rows ({len(ga4_available)/len(df)*100:.1f} %)")
print(f"Both IS TRUE: {len(both_available):,} rows ({len(both_available)/len(df)*100:.1f} %)")
print("Note: ~94 % of rows have no GA4 data in March — many clients' GA4 starts later.")

gsc_data_available IS TRUE: 3,611,061 rows (36.7 %)
ga4_data_available IS TRUE: 413,966 rows (4.2 %)
Both IS TRUE: 364,347 rows (3.7 %)
Note: ~94 % of rows have no GA4 data in March — many clients' GA4 starts later.


## **Five-feature frame (March 2026)**

Build a small feature frame. Every feature has one line: "knowable at the decision moment because…"


In [16]:
# Build 5-feature frame: 7-day trailing averages + content_type
# Work on a manageable sample for speed
sample_content = df["content_hash_id"].sample(n=5000, random_state=42)
sample_df = df[df["content_hash_id"].isin(sample_content)].copy()
sample_df = sample_df.sort_values(["content_hash_id", "report_date"])

# 7-day trailing averages (knowable at decision moment because they only use past data)
sample_df["feat_impr_7d"] = (
    sample_df.groupby("content_hash_id")["gsc_impressions"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
sample_df["feat_clicks_7d"] = (
    sample_df.groupby("content_hash_id")["gsc_clicks"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
sample_df["feat_pos_7d"] = (
    sample_df.groupby("content_hash_id")["gsc_avg_position"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)
sample_df["feat_scroll_7d"] = (
    sample_df.groupby("content_hash_id")["scroll_events"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Join content_type from dim_content (static — knowable because it doesn't change)
sample_df = sample_df.merge(
    dim_df[["content_hash_id", "content_type"]],
    on="content_hash_id",
    how="left"
)

feature_frame = sample_df[
    ["report_date", "client_hash_id", "content_hash_id",
     "feat_impr_7d", "feat_clicks_7d", "feat_pos_7d",
     "feat_scroll_7d", "content_type"]
].copy()

print(f"Feature frame: {len(feature_frame):,} rows")
print()
print("Feature availability:")
print("  feat_impr_7d  — knowable at decision moment because it uses only the prior 7 days of GSC impressions.")
print("  feat_clicks_7d — knowable at decision moment because it uses only the prior 7 days of GSC clicks.")
print("  feat_pos_7d   — knowable at decision moment because it uses only the prior 7 days of avg position.")
print("  feat_scroll_7d — knowable at decision moment because it uses only the prior 7 days of scroll events.")
print("  content_type  — knowable at decision moment because it is static metadata (does not change over time).")
print()
print(feature_frame.head(5).to_string())

Feature frame: 150,937 rows

Feature availability:
  feat_impr_7d  — knowable at decision moment because it uses only the prior 7 days of GSC impressions.
  feat_clicks_7d — knowable at decision moment because it uses only the prior 7 days of GSC clicks.
  feat_pos_7d   — knowable at decision moment because it uses only the prior 7 days of avg position.
  feat_scroll_7d — knowable at decision moment because it uses only the prior 7 days of scroll events.
  content_type  — knowable at decision moment because it is static metadata (does not change over time).

  report_date           client_hash_id           content_hash_id  feat_impr_7d  feat_clicks_7d  feat_pos_7d  feat_scroll_7d     content_type
0  2026-03-01  client_73cda7b4e4f265ea  content_000e683801cdf58b      0.000000             0.0          NaN             NaN  keyword article
1  2026-03-02  client_73cda7b4e4f265ea  content_000e683801cdf58b      1.500000             0.0    20.000000             NaN  keyword article
2  2026-03-0

### **The trap: deliberate leakage experiment**

Add ONE label-derived column, watch the score jump, then remove it.

In [17]:
# Build the honest label: forward 30-day impression decline (>20 % drop)
sample_df["forward_impr"] = (
    sample_df.groupby("content_hash_id")["gsc_impressions"]
    .transform(lambda x: x.shift(-30))
)
sample_df["forward_decline"] = (
    (sample_df["gsc_impressions"] - sample_df["forward_impr"])
    / sample_df["gsc_impressions"].clip(lower=1)
).clip(lower=0) > 0.20

# Merge label into feature frame
feature_frame = feature_frame.merge(
    sample_df[["report_date", "client_hash_id", "content_hash_id", "forward_decline"]],
    on=["report_date", "client_hash_id", "content_hash_id"],
    how="left"
)

# --- HONEST score (random baseline: precision@K) ---
K = 100
pos_count = feature_frame["forward_decline"].sum()
neg_count = len(feature_frame) - pos_count

np.random.seed(42)
random_order = np.random.permutation(len(feature_frame))
topK = feature_frame.iloc[random_order[:K]]
honest_precision = topK["forward_decline"].mean()
print(f"Honest precision@{K} (random baseline): {honest_precision:.3f}")

# --- LEAKY score: add forward_decline itself as a feature ---
feature_frame["LEAK_forward_decline"] = feature_frame["forward_decline"]
leaky_features = feature_frame.sort_values(
    ["LEAK_forward_decline", "feat_impr_7d"],
    ascending=[False, True]
)
topK_leaky = leaky_features.head(K)
leaky_precision = topK_leaky["forward_decline"].mean()
print(f"Leaky precision@{K} (with label-derived column): {leaky_precision:.3f}")
print(f"Score jumped by {(leaky_precision - honest_precision):.3f} — classic leakage.")

# --- Remove the leaky column, keep the honest number ---
del feature_frame["LEAK_forward_decline"]
print(f"\nLeaky column removed. Keeping honest precision@{K}: {honest_precision:.3f}")

Honest precision@100 (random baseline): 0.000
Leaky precision@100 (with label-derived column): 1.000
Score jumped by 1.000 — classic leakage.

Leaky column removed. Keeping honest precision@100: 0.000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation** — Unbalanced client history: Different clients have different GSC/GA4 start dates. In March 2026, ~94 % of rows have no GA4 data. This means any GA4-based feature is only knowable for a small subset of content-items, creating an unbalanced panel. A model trained on the full month will either drop GA4 features (losing signal for the minority) or impute them (adding noise for the majority). The feature frame should filter to ga4_data_available IS TRUE whenever GA4 features are used, and report the reduced row count transparently.

In [18]:
# Verify: show the GA4 gap per client
ga4_start_counts = df.groupby("client_hash_id")["ga4_data_available"].sum()
clients_with_any_ga4 = (ga4_start_counts > 0).sum()
clients_without_ga4 = (ga4_start_counts == 0).sum()
print(f"Clients with any GA4 in March 2026: {clients_with_any_ga4}")
print(f"Clients with no GA4 in March 2026: {clients_without_ga4}")
print(f"GA4 coverage: {clients_with_any_ga4 / (clients_with_any_ga4 + clients_without_ga4) * 100:.1f} % of clients")

Clients with any GA4 in March 2026: 41
Clients with no GA4 in March 2026: 14
GA4 coverage: 74.5 % of clients


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.